# On-ramp: one complete training loop

This notebook isolates the mechanics assumed by *Making It Trainable*: tensor shapes, a scalar loss, reverse accumulation, mini-batching, and parameter updates. It deliberately uses one linear map so architecture cannot hide the contract.

In [1]:
import numpy as np

features = np.array([
    [1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, -1.0],
    [-1.0, 2.0], [0.5, -0.5], [-0.5, 0.5], [2.0, 2.0],
])
targets = 2.0 * features[:, :1] - features[:, 1:2] + 0.5
weight = np.zeros((2, 1))
bias = np.zeros(1)

assert features.shape == (8, 2)
assert targets.shape == (8, 1)
print(f"features={tuple(features.shape)} targets={tuple(targets.shape)}")

features=(8, 2) targets=(8, 1)


## The contract before the loop

The full objective is the mean squared residual over all eight observations. A batch mean estimates it only relative to a declared sampling rule. Below, a seeded permutation supplies mini-batches of four without replacement within each epoch.

In [2]:
learning_rate = 0.15
batch_size = 4
generator = np.random.default_rng(6210)
loss_history = []

for epoch in range(25):
    order = generator.permutation(len(features))
    for start in range(0, len(features), batch_size):
        index = order[start : start + batch_size]
        x_batch, y_batch = features[index], targets[index]

        prediction = x_batch @ weight + bias
        residual = prediction - y_batch
        loss = np.mean(residual**2)
        assert np.ndim(loss) == 0

        weight_gradient = (2 / len(index)) * x_batch.T @ residual
        bias_gradient = (2 / len(index)) * residual.sum(axis=0)
        assert weight_gradient.shape == weight.shape

        weight -= learning_rate * weight_gradient
        bias -= learning_rate * bias_gradient
        loss_history.append(float(loss))

print(f"weight={weight.ravel().tolist()} bias={bias.item():.4f}")

weight=[1.9998335114994545, -1.0001653888970106] bias=0.5004


## Readiness checks

Before Chapter 1, answer these without changing the code: Which axes does `mean()` reduce? Which population does the batch loss target? Where does reverse accumulation appear in the two gradient formulas? Why must the update occur after those gradients are computed? Then change the batch size and learning rate, predict the qualitative effect, and rerun.

In [3]:
full_prediction = features @ weight + bias
full_loss = np.mean((full_prediction - targets) ** 2)

assert full_loss < 1e-6
assert np.allclose(weight, np.array([[2.0], [-1.0]]), atol=2e-3)
assert np.allclose(bias, np.array([0.5]), atol=2e-3)
print(f"full_loss={full_loss:.3e}")

full_loss=6.727e-08
